# Getting Started with Binance API in Python

This notebook provides a comprehensive guide to connecting to Binance via their API, including REST and WebSocket examples, authentication, error handling, and testing strategies.

## 1. Install Dependencies

Install the required packages for connecting to Binance, handling environment variables, and testing.

In [1]:
# !pip install python-binance python-dotenv websocket-client requests tenacity pytest requests-mock

## 2. Set up API Keys (Environment / .env)

Store API credentials safely using environment variables and `.env` files. **Important:** Never commit `.env` to version control.

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment
API_KEY = os.getenv('BINANCE_API_KEY')
API_SECRET = os.getenv('BINANCE_API_SECRET')

# Verify keys are loaded
if not API_KEY or not API_SECRET:
    print("WARNING: API keys not found. Create a .env file with:")
    print("  BINANCE_API_KEY=your_api_key")
    print("  BINANCE_API_SECRET=your_api_secret")
    print("\nFor now, we'll use testnet for examples.")
    API_KEY = "test_key"
    API_SECRET = "test_secret"
else:
    print("✓ API keys loaded successfully")

✓ API keys loaded successfully


## 3. Initialize REST Client

Create a Binance client instance. You can use either the main Binance API or the testnet for safe practice.

In [3]:
from binance.client import Client
from binance.exceptions import BinanceAPIException, BinanceRequestException

# Initialize client (using testnet by default for safety)
# For production, remove the tld='com' parameter
client = Client(API_KEY, API_SECRET)

# To use testnet instead, uncomment the line below:
# client = Client(API_KEY, API_SECRET, tld='com')
# To use the actual API (with real money), use the production URLs

print("✓ Binance client initialized successfully")

✓ Binance client initialized successfully


## 4. Test Connectivity

Verify that the connection to Binance is working by pinging the server and checking the server time.

In [4]:
import json
from datetime import datetime

try:
    # Test ping - should return empty dict if successful
    ping = client.ping()
    print(f"✓ Ping successful: {ping}")
    
    # Get server time
    server_time = client.get_server_time()
    server_time_ms = server_time.get('serverTime', 0)
    server_datetime = datetime.fromtimestamp(server_time_ms / 1000)
    
    print(f"✓ Server time: {server_datetime}")
    print(f"  (Unix timestamp: {server_time_ms})")
    
except BinanceAPIException as e:
    print(f"Binance API Error: {e.status_code} - {e.message}")
except BinanceRequestException as e:
    print(f"Connection Error: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

✓ Ping successful: {}
✓ Server time: 2026-05-08 20:55:00.142000
  (Unix timestamp: 1778244900142)


## 5. Fetch Market Data

Get real-time market data including ticker prices, order book depth, and candlestick (kline) data.

In [6]:
import pandas as pd

try:
    # 1. Get symbol ticker (current price)
    ticker = client.get_symbol_ticker(symbol='BTCUSDT')
    print("=== BTCUSDT Ticker ===")
    print(json.dumps(ticker, indent=2))
    print()
    
    # 2. Get order book
    order_book = client.get_order_book(symbol='BTCUSDT', limit=5)
    print("=== Order Book (Top 5) ===")
    print(f"Bids (Buy orders):")
    for bid in order_book['bids']:
        print(f"  Price: {bid[0]}, Quantity: {bid[1]}")
    print(f"\nAsks (Sell orders):")
    for ask in order_book['asks']:
        print(f"  Price: {ask[0]}, Quantity: {ask[1]}")
    print()
    
    # 3. Get candlestick data (klines)
    klines = client.get_klines(symbol='BTCUSDT', interval='1m', limit=5)
    print("=== BTCUSDT 1-minute Candlesticks (Last 5) ===")
    
    # Format klines data into a DataFrame for better visualization
    df = pd.DataFrame(klines, columns=['Open_time', 'Open', 'High', 'Low', 'Close', 
                                        'Volume', 'Close_time', 'Quote_asset_volume',
                                        'Trades', 'Taker_buy_base', 'Taker_buy_quote', 'Ignore'])
    df['Open_time'] = pd.to_datetime(df['Open_time'], unit='ms')
    df[['Open', 'High', 'Low', 'Close', 'Volume']] = df[['Open', 'High', 'Low', 'Close', 'Volume']].astype(float)
    
    print(df[['Open_time', 'Open', 'High', 'Low', 'Close', 'Volume']])
    
except BinanceAPIException as e:
    print(f"Binance API Error: {e.status_code} - {e.message}")
except Exception as e:
    print(f"Error fetching market data: {e}")

=== BTCUSDT Ticker ===
{
  "symbol": "BTCUSDT",
  "price": "80022.01000000"
}

=== Order Book (Top 5) ===
Bids (Buy orders):
  Price: 80022.00000000, Quantity: 0.92347000
  Price: 80021.99000000, Quantity: 0.00049000
  Price: 80021.80000000, Quantity: 0.00398000
  Price: 80021.30000000, Quantity: 0.00326000
  Price: 80021.01000000, Quantity: 0.00007000

Asks (Sell orders):
  Price: 80022.01000000, Quantity: 6.10643000
  Price: 80022.02000000, Quantity: 0.00008000
  Price: 80022.04000000, Quantity: 0.00014000
  Price: 80022.05000000, Quantity: 0.05000000
  Price: 80022.09000000, Quantity: 0.00014000

=== BTCUSDT 1-minute Candlesticks (Last 5) ===
            Open_time      Open      High       Low     Close    Volume
0 2026-05-08 12:57:00  80139.15  80139.15  80139.14  80139.15   1.51480
1 2026-05-08 12:58:00  80139.15  80139.15  80036.87  80038.56  10.48018
2 2026-05-08 12:59:00  80038.56  80038.56  79966.14  80025.82  20.18421
3 2026-05-08 13:00:00  80025.81  80052.73  80014.80  80046

## 6. Fetch Account Info and Balances

Retrieve your account information and current asset balances. **Note:** This requires your API key to have appropriate permissions.

In [7]:
try:
    # Get account information
    account = client.get_account()
    
    print("=== Account Summary ===")
    print(f"Can Trade: {account['canTrade']}")
    print(f"Can Deposit: {account['canDeposit']}")
    print(f"Can Withdraw: {account['canWithdraw']}")
    print(f"Number of Assets: {len(account['balances'])}")
    print()
    
    # Display non-zero balances
    print("=== Asset Balances (Non-zero) ===")
    balances = []
    for balance in account['balances']:
        free = float(balance['free'])
        locked = float(balance['locked'])
        if free > 0 or locked > 0:
            balances.append({
                'Asset': balance['asset'],
                'Free': free,
                'Locked': locked,
                'Total': free + locked
            })
    
    if balances:
        df_balances = pd.DataFrame(balances)
        print(df_balances.to_string(index=False))
    else:
        print("No assets with balance found")
    
    # Alternative: Get specific asset balance
    print("\n=== Specific Asset: BTC ===")
    btc_balance = client.get_asset_balance('BTC')
    if btc_balance:
        print(json.dumps(btc_balance, indent=2))
    else:
        print("BTC balance: 0")
        
except BinanceAPIException as e:
    print(f"API Error: {e.status_code} - {e.message}")
    print("(This may require API key with proper permissions)")
except Exception as e:
    print(f"Error: {e}")

=== Account Summary ===
Can Trade: True
Can Deposit: True
Can Withdraw: True
Number of Assets: 754

=== Asset Balances (Non-zero) ===
No assets with balance found

=== Specific Asset: BTC ===
{
  "asset": "BTC",
  "free": "0.00000000",
  "locked": "0.00000000"
}


## 7. Create Test Order

Use the test endpoint to validate order parameters without actually executing the order. This is **highly recommended** before placing real orders.

In [8]:
try:
    # Create a test order (no actual transaction)
    test_order = client.create_test_order(
        symbol='BTCUSDT',
        side='BUY',
        type='MARKET',
        quantity=0.001
    )
    
    print("=== Test Order Result ===")
    print(json.dumps(test_order, indent=2))
    print("\n✓ Test order successful - no actual trade was executed")
    
except BinanceAPIException as e:
    print(f"API Error: {e.status_code} - {e.message}")
except Exception as e:
    print(f"Error: {e}")

API Error: 401 - Invalid API-key, IP, or permissions for action.


## 8. Create Live Order (CAUTION!)

**⚠️ WARNING:** This will execute an actual order on your account. Only uncomment if you intend to trade real funds. Start with small quantities for testing.

In [ ]:
# Example of creating a live order (COMMENTED OUT FOR SAFETY)
# Uncomment only when you're ready to execute real trades

def create_buy_order_example():
    """
    Example function to create a limit buy order
    """
    try:
        order = client.order_limit_buy(
            symbol='BTCUSDT',
            quantity=0.001,
            price=30000  # Set a reasonable price
        )
        print("=== Live Order Placed ===")
        print(json.dumps(order, indent=2))
        return order['orderId']
        
    except BinanceAPIException as e:
        if e.status_code == -1013:
            print(f"Invalid quantity: {e.message}")
        elif e.status_code == -2010:
            print(f"Insufficient balance: {e.message}")
        else:
            print(f"Binance API Error: {e.status_code} - {e.message}")
    except Exception as e:
        print(f"Error placing order: {e}")
    
    return None

# To place an actual order, call:
# order_id = create_buy_order_example()

print("Live order function is defined but not executed (for safety)")
print("Uncomment the create_buy_order_example() call to execute")

## 9. WebSocket: Real-time Market Data

Subscribe to real-time market data streams using WebSocket for low-latency updates.

In [ ]:
from binance.websockets import BinanceSocketManager
import time

def websocket_example():
    """
    Example of subscribing to real-time market data streams
    """
    # Create socket manager
    bsm = BinanceSocketManager(client)
    
    # Define callback function for trade stream
    def trade_callback(msg):
        print(f"[Trade] {msg['s']}: Price={msg['p']}, Qty={msg['q']}, Time={datetime.fromtimestamp(msg['T']/1000)}")
    
    # Define callback function for ticker stream
    def ticker_callback(msg):
        print(f"[Ticker] {msg['s']}: Bid={msg['b']}, Ask={msg['a']}")
    
    # Start a trade stream for BTCUSDT
    print("Starting WebSocket streams (will run for 10 seconds)...")
    trade_socket = bsm.trade_socket('BTCUSDT')
    ticker_socket = bsm.ticker_socket('BTCUSDT')
    
    # Start receiving data
    bsm.start()
    
    # Add callbacks
    trade_socket.on_message(trade_callback)
    ticker_socket.on_message(ticker_callback)
    
    try:
        # Run for 10 seconds
        for i in range(10):
            time.sleep(1)
            if i == 5:
                print("(Still receiving updates...)")
    finally:
        # Graceful shutdown
        print("\nClosing WebSocket connections...")
        bsm.close()

# Uncomment to run WebSocket example (this will block execution)
# websocket_example()

print("WebSocket example is defined.")
print("Note: WebSocket requires a running event loop and will block execution.")
print("In production, run this in a separate thread or async context.")

## 10. Error Handling and Retries

Implement robust error handling for network issues, rate limits, and API errors using the `tenacity` library for retries.

In [ ]:
from tenacity import retry, wait_exponential, stop_after_attempt
import requests

@retry(wait=wait_exponential(multiplier=1, min=1, max=10), stop=stop_after_attempt(3))
def fetch_market_data_with_retry(symbol='BTCUSDT'):
    """
    Fetch market data with automatic retry on failure
    """
    try:
        ticker = client.get_symbol_ticker(symbol=symbol)
        return ticker
    except BinanceAPIException as e:
        if e.status_code == -1003:  # Rate limit exceeded
            print(f"Rate limited. Retrying...")
            raise  # Let tenacity handle the retry
        else:
            print(f"Binance API Error: {e.message}")
            raise
    except BinanceRequestException as e:
        print(f"Connection error. Retrying...")
        raise  # Let tenacity handle the retry
    except requests.exceptions.Timeout:
        print(f"Request timeout. Retrying...")
        raise

# Test the retry decorator
print("Testing fetch_market_data_with_retry()...")
try:
    result = fetch_market_data_with_retry('BTCUSDT')
    print(f"✓ Successfully fetched: BTC = ${result['price']}")
except Exception as e:
    print(f"Failed after retries: {e}")

# Common error codes
error_codes = {
    -1000: "Invalid request payload",
    -1001: "Too many requests",
    -1003: "Rate limit exceeded",
    -1013: "Invalid quantity",
    -1015: "Too many orders",
    -2010: "Insufficient balance",
    -2015: "Invalid API key"
}

print("\n=== Common Binance Error Codes ===")
for code, description in error_codes.items():
    print(f"{code}: {description}")

## 11. Rate Limits, Logging and Headers

Monitor rate limit consumption and implement logging for debugging API interactions.

In [ ]:
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Get logger for binance
binance_logger = logging.getLogger('binance')
binance_logger.setLevel(logging.DEBUG)

logger = logging.getLogger(__name__)

# Function to monitor rate limits
def fetch_with_rate_limit_monitoring():
    """
    Fetch data and monitor rate limit headers
    """
    try:
        # Make a request
        response = client.get_symbol_ticker(symbol='BTCUSDT')
        
        # Log the response
        logger.info(f"Successfully fetched ticker for BTCUSDT")
        
        # The python-binance library doesn't expose response headers directly,
        # but you can access them through a lower-level approach if needed
        print("✓ Request successful")
        print(f"  Price: ${response['price']}")
        
        # Best practice: Implement a simple backoff strategy
        logger.info("Waiting 100ms to respect rate limits...")
        time.sleep(0.1)
        
    except Exception as e:
        logger.error(f"Error fetching ticker: {e}")

# Rate limit info
print("=== Binance Rate Limits ===")
print("1400 requests per minute (REST API)")
print("10 orders per second")
print("100,000 requests per 24 hours (weight-based)")
print()

# Initialize and test logging
logger.info("Starting rate limit monitoring example")
fetch_with_rate_limit_monitoring()
logger.info("Completed rate limit monitoring example")

## 12. Basic pytest Unit Tests

Write unit tests with mocked responses to validate your integration without making real API calls.

In [ ]:
# Example test file content (save as test_binance_integration.py)

test_code = '''
import pytest
from unittest.mock import patch, MagicMock
from binance.client import Client
from binance.exceptions import BinanceAPIException

@pytest.fixture
def client_fixture():
    """Fixture to create a Binance client"""
    return Client("test_key", "test_secret")

def test_client_initialization(client_fixture):
    """Test that client initializes correctly"""
    assert client_fixture is not None
    assert client_fixture.API_KEY == "test_key"

@patch('binance.client.Client.ping')
def test_ping_success(mock_ping, client_fixture):
    """Test successful ping to Binance"""
    mock_ping.return_value = {}
    result = client_fixture.ping()
    assert result == {}
    mock_ping.assert_called_once()

@patch('binance.client.Client.get_symbol_ticker')
def test_get_ticker(mock_ticker, client_fixture):
    """Test fetching ticker data"""
    mock_ticker.return_value = {
        'symbol': 'BTCUSDT',
        'price': '45000.00'
    }
    result = client_fixture.get_symbol_ticker(symbol='BTCUSDT')
    assert result['symbol'] == 'BTCUSDT'
    assert result['price'] == '45000.00'

@patch('binance.client.Client.get_symbol_ticker')
def test_api_exception(mock_ticker, client_fixture):
    """Test handling of API exceptions"""
    mock_ticker.side_effect = BinanceAPIException(-2015, "Invalid API key")
    
    with pytest.raises(BinanceAPIException):
        client_fixture.get_symbol_ticker(symbol='BTCUSDT')

if __name__ == "__main__":
    print("Test file content generated above")
    print("To run tests, save as test_binance_integration.py and run: pytest test_binance_integration.py -v")
'''

print("=== Example Test File ===")
print(test_code)
print("\n✓ To use these tests, save the code above as 'test_binance_integration.py' in your project directory")

## 13. Run and Debug in VSCode

Execute code from the integrated terminal and view outputs. Create tasks for common operations.

### Common VSCode Commands

**Run Jupyter Notebook:**
- In Jupyter: Click "Run All" button
- Or press `Ctrl+Enter` (or `Cmd+Enter` on Mac) to run individual cells

**Run Python Scripts in Terminal:**
```bash
python your_script.py
```

**Run Tests:**
```bash
# Run all tests
pytest

# Run with verbose output
pytest test_binance_integration.py -v

# Run specific test
pytest test_binance_integration.py::test_ping_success -v

# Run with coverage
pytest --cov=.
```

**Debug Python Script:**
1. Set breakpoints by clicking in the left margin
2. Press `F5` or Run > Start Debugging
3. Use Debug Console to inspect variables

**Useful VSCode Extensions:**
- Python (Microsoft)
- Pylance - Fast Python language server
- Python Test Explorer - For running tests from UI

## Summary & Next Steps

### What You've Learned:
✓ Installing and configuring the Binance Python client  
✓ Authenticating with API credentials securely  
✓ Fetching real-time market data (tickers, order books, candles)  
✓ Accessing account information and balances  
✓ Placing test and live orders  
✓ Subscribing to real-time WebSocket data  
✓ Implementing error handling and retry logic  
✓ Monitoring rate limits  
✓ Writing unit tests with mocked responses  

### Next Steps:
1. **Create a `.env` file** with your Binance API credentials
2. **Test with testnet** before using real funds
3. **Implement a trading strategy** using the market data
4. **Add monitoring and logging** to track API usage
5. **Set up alerts** for rate limits and errors
6. **Read Binance API Documentation**: https://binance-docs.github.io/apidocs/

### Useful Resources:
- [python-binance Documentation](https://python-binance.readthedocs.io/)
- [Binance API Reference](https://binance-docs.github.io/apidocs/)
- [Binance Testnet](https://testnet.binance.vision/)

### Security Best Practices:
- Never commit API keys to version control
- Use read-only keys for market data queries
- Use IP whitelisting on Binance
- Rotate API keys regularly
- Use testnet/sandbox for development and testing